In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)

In [0]:

from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/employees_delta"
)

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql DESCRIBE HISTORY employees

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T09:40:00.000Z,143740682961545,azuser7223_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(78140097717768),3cdc6f7e-1a4d-4e06-ab3f-97b0da687fdd,0617-093022-3997b0q9-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1160)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/employees_delta")
 
display(df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql OPTIMIZE employees

path,metrics
abfss://unity-catalog-storage@dbstoragehp53zfrbdouli.dfs.core.windows.net/7405611601997461/__unitystorage/catalogs/0b1a0499-46fb-4d22-944c-4a0cb2287091/tables/a8d967c4-f706-497e-aed6-5da35f788e41,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781690539257, 1781690539889, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql OPTIMIZE employees
ZORDER BY(salary)

path,metrics
abfss://unity-catalog-storage@dbstoragehp53zfrbdouli.dfs.core.windows.net/7405611601997461/__unitystorage/catalogs/0b1a0499-46fb-4d22-944c-4a0cb2287091/tables/a8d967c4-f706-497e-aed6-5da35f788e41,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1160), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781690946799, 1781690947399, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql VACUUM employees

path
abfss://unity-catalog-storage@dbstoragehp53zfrbdouli.dfs.core.windows.net/7405611601997461/__unitystorage/catalogs/0b1a0499-46fb-4d22-944c-4a0cb2287091/tables/a8d967c4-f706-497e-aed6-5da35f788e41
